# Rebuilding the labels and the frames from the audio (level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/extract.ipynb)

The routers learn from two things per corpus and expert: the expert's **labels** (its transcripts, their word-error
counters against the reference, and the mean of its encoder states) and its **frames** (the frame-level encoder states
themselves, fp16, at the encoder's own frame rate). Levels 1 and 2 read both from the published dataset; this
notebook rebuilds them from the audio: every expert of every corpus's trio decodes the corpus in full and its encoder
states are stored.

**Where it writes.** A local folder, `$HITASR_CACHE/rebuild` (on Colab `/content/hitasr_cache/rebuild`). Then run
`main_<corpus>` at level 3 **on the same machine**: it reads the labels and frames from that folder, the published
searches from the public dataset, and retrains every fold. Nothing is uploaded unless you set `PUSH_TO` to a dataset
repository of your own (that, and only that, needs a token with write access; level 3 elsewhere then reads it with
`HITASR_HUB=<that repository>` set before the imports).

**Token.** Cohere Transcribe (in the trios of AMI, Earnings-22 and AfriSpeech) is gated on the Hub: accept its terms
with your account, then give this notebook a **read** token as the Colab secret `HF_TOKEN` (or the environment
variable of that name). The other experts and the audio need none.

**Cost.** Each (corpus, expert) is one pass of the model over the corpus: minutes to an hour on a 96 GB GPU (a
smaller GPU gets smaller batches), with 1-13 GB of frames each; a corpus's trio takes 7-18 GB of disk. The audio is
read from the published `<corpus>_base` configurations (16 kHz, the clips the paper uses). An expert whose labels and
frames are already complete in the folder is skipped, so an interrupted run resumes.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets<4.0", "jiwer", "librosa", "soundfile",
                "transformers>=4.57", "accelerate", "sentencepiece", "torchaudio", "mistral-common[audio]", "peft"],
               check=True)

In [ ]:
CORPORA = ("ami_sdm", "earnings22", "peoples_speech", "afrispeech")   # or a subset, e.g. ("earnings22",)
PUSH_TO = ""      # optional: "<you>/hit-asr-rebuild", a dataset repository of yours to upload the rebuild to as well

from hitasr.configs import MAIN, REBUILD
from hitasr.core import rebuild_dir, use_hub
from hitasr.extractor import compare_with_published, rebuild_plan, sweep
from hitasr.models.registry import load_expert_classes
from labkit.env import describe_env, login_hf

OUT = rebuild_dir()                                  # where main_<corpus> reads them at level 3
use_hub(f"local:{OUT}", records=f"local:{OUT}")      # labels, frames and the extraction records all go there
login_hf(required=any(m in REBUILD.gated for c in CORPORA for m in MAIN[c].members))   # a read token (gated experts)
describe_env()
print(f"writing to {OUT}")

## 1. What to rebuild

Every expert of each corpus's trio (`hitasr.configs.MAIN[corpus].members`) not yet complete in the folder.

In [ ]:
plan = rebuild_plan(CORPORA)

## 2. The pass

Per corpus the audio is loaded once; then per expert a short smoke test, the batch fit (the longest clips are pushed
through once and the batch is halved until they fit), the pass, and the store written to the folder. Every clip is
decoded; nothing is adopted from an existing label set.

In [ ]:
log = sweep(plan, load_expert_classes(), REBUILD.batch, min_batch=REBUILD.min_batch, decode="full")
log

## 3. Check against the published pass

The same checkpoint, decoder and normaliser give the same transcripts up to the GPU's numerics: nearly every
normalised transcript is identical, a few flip, and each expert's corpus WER matches the published labels to about the
third decimal.

In [ ]:
compare_with_published(log)

## 4. Upload (optional)

Only with `PUSH_TO` set: the folder goes to that dataset repository (created private if it does not exist) with a
write token.

In [ ]:
if PUSH_TO:
    from huggingface_hub import HfApi
    login_hf()                                       # a token with write access to PUSH_TO
    HfApi().create_repo(PUSH_TO, repo_type="dataset", private=True, exist_ok=True)
    HfApi().upload_large_folder(repo_id=PUSH_TO, repo_type="dataset", folder_path=str(OUT))
    print(f"uploaded {OUT} to {PUSH_TO}: run main_<corpus> at level 3 elsewhere with HITASR_HUB={PUSH_TO}")